# 05. Baseline Robustness

So sánh baseline trên clean test và các biến thể noisy test.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"
PREDICTIONS_DIR = ROOT / "outputs" / "predictions"
REPORTS_DIR = ROOT / "outputs" / "reports"
METRICS_DIR = ROOT / "outputs" / "metrics"

## 2. Kiểm tra output bắt buộc

Các file này phải tồn tại sau khi chạy:

```powershell
python scripts/evaluate_baseline_robustness.py
```

In [ ]:
required_files = {
    "robustness_results": TABLES_DIR / "baseline_robustness_results.csv",
    "robustness_drop": TABLES_DIR / "baseline_robustness_drop.csv",
    "class_report": TABLES_DIR / "baseline_robustness_class_report.csv",
    "predictions": PREDICTIONS_DIR / "baseline_robustness_predictions.csv",
    "metrics_json": METRICS_DIR / "baseline_robustness_results.json",
    "report": REPORTS_DIR / "baseline_robustness_report.md",
    "macro_f1_sentiment": FIGURES_DIR / "baseline_robustness_macro_f1_sentiment.png",
    "macro_f1_topic": FIGURES_DIR / "baseline_robustness_macro_f1_topic.png",
    "macro_f1_drop_sentiment": FIGURES_DIR / "baseline_robustness_macro_f1_drop_sentiment.png",
    "macro_f1_drop_topic": FIGURES_DIR / "baseline_robustness_macro_f1_drop_topic.png",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required Stage 5 files: {missing}")

print("All required Stage 5 files exist.")

## 3. Load kết quả

In [ ]:
results_df = pd.read_csv(TABLES_DIR / "baseline_robustness_results.csv")
drop_df = pd.read_csv(TABLES_DIR / "baseline_robustness_drop.csv")
class_report_df = pd.read_csv(TABLES_DIR / "baseline_robustness_class_report.csv")

display(results_df.head())
display(drop_df.head())
display(class_report_df.head())

## 4. Macro-F1 theo variant

Nội dung: xem baseline thay đổi thế nào khi test text bị nhiễu.

In [ ]:
display(Image(filename=str(FIGURES_DIR / "baseline_robustness_macro_f1_sentiment.png")))
display(Image(filename=str(FIGURES_DIR / "baseline_robustness_macro_f1_topic.png")))

## 5. Macro-F1 drop theo variant

Drop được tính theo công thức:

```text
macro_f1_drop = clean_macro_f1 - variant_macro_f1
```

Nếu drop lớn, model kém bền trước loại noise đó.

In [ ]:
display(Image(filename=str(FIGURES_DIR / "baseline_robustness_macro_f1_drop_sentiment.png")))
display(Image(filename=str(FIGURES_DIR / "baseline_robustness_macro_f1_drop_topic.png")))

## 6. Bảng kết quả tổng hợp

In [ ]:
variant_order = [
    "clean",
    "typo_light",
    "typo_medium",
    "teencode_light",
    "mixed_light",
    "no_accent",
    "mixed_no_accent",
]

results_view = results_df.copy()
results_view["variant"] = pd.Categorical(results_view["variant"], categories=variant_order, ordered=True)
results_view = results_view.sort_values(["task", "model", "variant"])

display(results_view)

## 7. Bảng drop tổng hợp

In [ ]:
drop_view = drop_df.copy()
drop_view["variant"] = pd.Categorical(drop_view["variant"], categories=variant_order, ordered=True)
drop_view = drop_view.sort_values(["task", "model", "variant"])

display(drop_view)

## 8. Clean baseline reference

Nội dung: xác định baseline mạnh nhất trên clean test cho từng task.

In [ ]:
clean_results = results_df[results_df["variant"] == "clean"].copy()
clean_results = clean_results.sort_values(["task", "macro_f1"], ascending=[True, False])

display(clean_results)

best_clean_by_task = clean_results.groupby("task").head(1).reset_index(drop=True)
display(Markdown("### Best clean baseline by task"))
display(best_clean_by_task)

## 9. Worst robustness drop

Nội dung: xác định noise nào làm từng model giảm mạnh nhất.

In [ ]:
worst_drop = (
    drop_df[drop_df["variant"] != "clean"]
    .sort_values(["task", "model", "macro_f1_drop"], ascending=[True, True, False])
    .groupby(["task", "model"])
    .head(1)
    .reset_index(drop=True)
)

display(worst_drop[[
    "task",
    "model",
    "variant",
    "clean_macro_f1",
    "variant_macro_f1",
    "macro_f1_drop",
    "macro_f1_relative_drop_percent",
]])

## 10. Sentiment robustness analysis

In [ ]:
sentiment_results = results_df[results_df["task"] == "sentiment"].copy()
sentiment_drop = drop_df[drop_df["task"] == "sentiment"].copy()

display(Markdown("### Sentiment Macro-F1 by variant"))
display(
    sentiment_results.pivot_table(
        index="variant",
        columns="model",
        values="macro_f1"
    ).reindex(variant_order)
)

display(Markdown("### Sentiment Macro-F1 drop by variant"))
display(
    sentiment_drop.pivot_table(
        index="variant",
        columns="model",
        values="macro_f1_drop"
    ).reindex(variant_order)
)

### Nhận xét sentiment

Các điểm cần ghi nhận:

```text
- TF-IDF char SVM là baseline mạnh nhất trên clean sentiment theo Macro-F1.
- no_accent và mixed_no_accent làm giảm mạnh nhất.
- Char-level SVM bền hơn word-level SVM khi bị mất dấu.
- Với typo/teencode nhẹ, mức giảm nhỏ hơn nhiều so với no_accent.
- Lớp neutral vốn yếu từ clean test, cần tiếp tục theo dõi trong PhoBERT và Error Analysis.
```

## 11. Topic robustness analysis

In [ ]:
topic_results = results_df[results_df["task"] == "topic"].copy()
topic_drop = drop_df[drop_df["task"] == "topic"].copy()

display(Markdown("### Topic Macro-F1 by variant"))
display(
    topic_results.pivot_table(
        index="variant",
        columns="model",
        values="macro_f1"
    ).reindex(variant_order)
)

display(Markdown("### Topic Macro-F1 drop by variant"))
display(
    topic_drop.pivot_table(
        index="variant",
        columns="model",
        values="macro_f1_drop"
    ).reindex(variant_order)
)

### Nhận xét topic

Các điểm cần ghi nhận:

```text
- TF-IDF word SVM là baseline mạnh nhất trên clean topic.
- Với clean và noise nhẹ, word-level SVM thường tốt hơn char-level SVM.
- Khi mất dấu, char-level SVM bền hơn word-level SVM.
- no_accent và mixed_no_accent làm topic giảm rất mạnh.
- Topic phụ thuộc nhiều vào từ khóa nội dung nên word-level TF-IDF bị ảnh hưởng nặng khi token bị mất dấu.
- Lớp others và các lớp nhỏ cần được phân tích kỹ ở Error Analysis.
```

## 12. Per-class analysis trên clean và no_accent

Nội dung: kiểm tra lớp nào giảm mạnh nhất khi bỏ dấu.

In [ ]:
summary_labels = {"accuracy", "macro avg", "weighted avg"}

per_class = class_report_df[
    ~class_report_df["label"].isin(summary_labels)
].copy()

focus_variants = ["clean", "no_accent", "mixed_no_accent"]
per_class_focus = per_class[per_class["variant"].isin(focus_variants)].copy()

display(per_class_focus.sort_values(["task", "model", "label", "variant"]))

# So sánh clean vs no_accent theo F1
clean_pc = per_class[per_class["variant"] == "clean"][
    ["task", "model", "label", "f1_score"]
].rename(columns={"f1_score": "clean_f1"})

no_accent_pc = per_class[per_class["variant"] == "no_accent"][
    ["task", "model", "label", "f1_score"]
].rename(columns={"f1_score": "no_accent_f1"})

pc_drop = clean_pc.merge(no_accent_pc, on=["task", "model", "label"], how="inner")
pc_drop["f1_drop"] = pc_drop["clean_f1"] - pc_drop["no_accent_f1"]
pc_drop["relative_f1_drop_percent"] = pc_drop.apply(
    lambda row: 0.0 if row["clean_f1"] == 0 else round(row["f1_drop"] / row["clean_f1"] * 100, 4),
    axis=1,
)

display(Markdown("### Per-class F1 drop: clean → no_accent"))
display(pc_drop.sort_values("f1_drop", ascending=False))

## 13. Best baseline trên từng variant

Nội dung: chọn baseline đối chứng mạnh nhất cho từng điều kiện test.

In [ ]:
best_by_variant = (
    results_df.sort_values(["task", "variant", "macro_f1"], ascending=[True, True, False])
    .groupby(["task", "variant"])
    .head(1)
    .reset_index(drop=True)
)

best_by_variant["variant"] = pd.Categorical(best_by_variant["variant"], categories=variant_order, ordered=True)
best_by_variant = best_by_variant.sort_values(["task", "variant"])

display(best_by_variant[[
    "task",
    "variant",
    "model",
    "accuracy",
    "macro_f1",
    "weighted_f1",
]])

## 14. Kết luận Stage 5

Kết luận chính:

```text
1. Baseline truyền thống tương đối ổn định trước typo, teencode và mixed_light.
2. Bỏ dấu tiếng Việt là loại noise gây suy giảm mạnh nhất.
3. mixed_no_accent có tác động gần giống no_accent, cho thấy bỏ dấu là yếu tố phá hủy chính.
4. Char-level SVM bền hơn word-level SVM trong điều kiện mất dấu.
5. Word-level SVM vẫn mạnh hơn trên clean và một số noise nhẹ, đặc biệt với topic classification.
6. Majority baseline không đổi theo noise vì luôn dự đoán lớp đa số, nên không dùng để kết luận về robustness.
```

Mốc so sánh cho PhoBERT:

```text
Sentiment:
- Clean: so với TF-IDF char SVM
- No-accent / mixed-no-accent: so với TF-IDF char SVM

Topic:
- Clean: so với TF-IDF word SVM
- No-accent / mixed-no-accent: so với TF-IDF char SVM
```

In [ ]:
display(Markdown("### Final clean baseline reference"))
display(best_clean_by_task)

display(Markdown("### Final best baseline by variant"))
display(best_by_variant[[
    "task",
    "variant",
    "model",
    "macro_f1",
    "accuracy",
]])

## 15. Stage 5 status

Stage 5 hoàn thành khi có đủ:

```text
outputs/tables/baseline_robustness_results.csv
outputs/tables/baseline_robustness_drop.csv
outputs/tables/baseline_robustness_class_report.csv
outputs/predictions/baseline_robustness_predictions.csv
outputs/figures/baseline_robustness_macro_f1_sentiment.png
outputs/figures/baseline_robustness_macro_f1_topic.png
outputs/figures/baseline_robustness_macro_f1_drop_sentiment.png
outputs/figures/baseline_robustness_macro_f1_drop_topic.png
outputs/reports/baseline_robustness_report.md
```

Giai đoạn tiếp theo:

```text
Stage 6 — PhoBERT Fine-tuning on Kaggle
```

Mục tiêu Stage 6:

```text
- Fine-tune PhoBERT cho sentiment.
- Fine-tune PhoBERT cho topic.
- Evaluate clean test.
- Sau đó evaluate cùng noisy variants để so với baseline robustness.
```